# Federated Medical Diagnosis + XAI — Colab Training

**Before running:** `Runtime -> Change runtime type -> T4 GPU`

In [ ]:
# 1. Clone your repo (replace with your own GitHub URL after you push it)
!git clone https://github.com/YOUR_USERNAME/fl-medical-xai.git
%cd fl-medical-xai
!pip install -q -r requirements.txt

In [ ]:
# 2. Mount Google Drive (put your dataset zip in Drive first)
from google.colab import drive
drive.mount('/content/drive')

# Example: unzip a dataset you uploaded to Drive into data/
!unzip -q '/content/drive/MyDrive/chest_xray.zip' -d data/
!ls data/

In [ ]:
# 3. Train the centralized baseline (for comparison later)
%cd src
!python train_centralized.py --data_dir ../data/train --test_data_dir ../data/test --epochs 8

In [ ]:
# 4. Run the Federated Learning simulation (5 simulated hospitals, non-IID split)
!python server.py --mode simulation --data_dir ../data/train --test_data_dir ../data/test \
    --num_clients 5 --num_rounds 10 --num_classes 2

In [ ]:
# 5. Save training history to results/history.json (used by the frontend chart)
import json, os
os.makedirs('../results', exist_ok=True)
from server import HISTORY
with open('../results/history.json', 'w') as f:
    json.dump(HISTORY, f)
print(HISTORY)

In [ ]:
# 6. Rename the last round's checkpoint to the name backend/app.py expects
import glob, shutil
ckpts = sorted(glob.glob('../saved_models/global_model_round*.pth'))
shutil.copy(ckpts[-1], '../saved_models/global_model_final.pth')
print('Final model:', ckpts[-1])

In [ ]:
# 7. Generate a Grad-CAM example on a sample test image (sanity check)
import torch
from model import build_model
from gradcam import explain_image_file, save_overlay
import glob

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model(num_classes=2, pretrained=False, device=device)
model.load_state_dict(torch.load('../saved_models/global_model_final.pth', map_location=device))

sample_img = glob.glob('../data/test/*/*')[0]
result = explain_image_file(model, sample_img, class_names=['NORMAL', 'PNEUMONIA'], device=device)
save_overlay(result['heatmap'], '../results/sample_gradcam.png')
print(result['predicted_class'], result['confidence'])

In [ ]:
# 8. Download everything you need back to your local machine
from google.colab import files
!zip -r results_and_models.zip ../saved_models/global_model_final.pth ../results
files.download('results_and_models.zip')